<a href="https://colab.research.google.com/github/TAUforPython/Graph-MachineLearning/blob/main/example_Interactive_Graph_Visualisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# First, make sure you have the required imports
import json
import networkx as nx
from IPython.display import HTML, display

# Your create_d3_html function here (with the missing import added)
def create_d3_html(G, node_sizes):
    """Create an HTML file with D3.js visualization"""
    import json  # Add this import

    # Convert graph to JSON
    nodes = [{"id": node, "size": node_sizes.get(node, 1)} for node in G.nodes()]
    links = [{"source": u, "target": v, "weight": d.get('weight', 1)}
             for u, v, d in G.edges(data=True)]

    graph_data = {"nodes": nodes, "links": links}

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>ERD Visualization</title>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body {{ margin: 0; font-family: Arial, sans-serif; }}
            #controls {{
                position: fixed;
                top: 10px;
                right: 10px;
                background: white;
                padding: 15px;
                border-radius: 8px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.2);
                z-index: 1000;
            }}
            .control-btn {{
                display: block;
                width: 100%;
                margin: 5px 0;
                padding: 8px;
                background: #4CAF50;
                color: white;
                border: none;
                border-radius: 4px;
                cursor: pointer;
            }}
            .control-btn:hover {{ background: #45a049; }}
            .node {{
                stroke: #fff;
                stroke-width: 2px;
                cursor: pointer;
            }}
            .node:hover {{
                stroke: #333;
                stroke-width: 3px;
            }}
            .link {{
                stroke: #999;
                stroke-opacity: 0.6;
            }}
            .node-label {{
                font-size: 12px;
                pointer-events: none;
                text-shadow: 0 1px 0 #fff;
            }}
            #tooltip {{
                position: absolute;
                background: rgba(0,0,0,0.8);
                color: white;
                padding: 5px 10px;
                border-radius: 4px;
                font-size: 12px;
                pointer-events: none;
                opacity: 0;
                transition: opacity 0.2s;
            }}
        </style>
    </head>
    <body>
        <div id="controls">
            <h3>Controls</h3>
            <button class="control-btn" onclick="zoomIn()">Zoom In +</button>
            <button class="control-btn" onclick="zoomOut()">Zoom Out -</button>
            <button class="control-btn" onclick="resetView()">Reset View</button>
            <button class="control-btn" onclick="exportSVG()">Export SVG</button>
            <hr>
            <div id="stats"></div>
        </div>

        <div id="tooltip"></div>

        <svg width="100%" height="600px"></svg>

        <script>
            const graphData = {json.dumps(graph_data)};

            const width = window.innerWidth;
            const height = 600;  // Fixed height for Colab

            const svg = d3.select("svg");
            const container = svg.append("g");

            // Add zoom behavior
            const zoom = d3.zoom()
                .scaleExtent([0.1, 4])
                .on("zoom", (event) => {{
                    container.attr("transform", event.transform);
                }});

            svg.call(zoom);

            // Create force simulation
            const simulation = d3.forceSimulation(graphData.nodes)
                .force("link", d3.forceLink(graphData.links).id(d => d.id).distance(200))
                .force("charge", d3.forceManyBody().strength(-500))
                .force("center", d3.forceCenter(width / 2, height / 2))
                .force("collision", d3.forceCollide().radius(d => Math.sqrt(d.size) * 10));

            // Create links
            const link = container.append("g")
                .selectAll("line")
                .data(graphData.links)
                .enter().append("line")
                .attr("class", "link")
                .attr("stroke-width", d => Math.sqrt(d.weight));

            // Create nodes
            const node = container.append("g")
                .selectAll("circle")
                .data(graphData.nodes)
                .enter().append("circle")
                .attr("class", "node")
                .attr("r", d => Math.sqrt(d.size) * 10)
                .attr("fill", d => d3.interpolateViridis(d.size / 10))
                .call(d3.drag()
                    .on("start", dragstarted)
                    .on("drag", dragged)
                    .on("end", dragended))
                .on("mouseover", (event, d) => {{
                    d3.select("#tooltip")
                        .style("opacity", 1)
                        .style("left", (event.pageX + 10) + "px")
                        .style("top", (event.pageY - 20) + "px")
                        .html(`<strong>${{d.id}}</strong><br>Size: ${{d.size}}`);
                }})
                .on("mouseout", () => {{
                    d3.select("#tooltip").style("opacity", 0);
                }});

            // Add labels
            const label = container.append("g")
                .selectAll("text")
                .data(graphData.nodes)
                .enter().append("text")
                .attr("class", "node-label")
                .attr("dx", d => Math.sqrt(d.size) * 10 + 5)
                .attr("dy", 4)
                .text(d => d.id.length > 20 ? d.id.substring(0, 17) + "..." : d.id);

            // Update positions on simulation tick
            simulation.on("tick", () => {{
                link
                    .attr("x1", d => d.source.x)
                    .attr("y1", d => d.source.y)
                    .attr("x2", d => d.target.x)
                    .attr("y2", d => d.target.y);

                node
                    .attr("cx", d => d.x)
                    .attr("cy", d => d.y);

                label
                    .attr("x", d => d.x)
                    .attr("y", d => d.y);
            }});

            // Drag functions
            function dragstarted(event, d) {{
                if (!event.active) simulation.alphaTarget(0.3).restart();
                d.fx = d.x;
                d.fy = d.y;
            }}

            function dragged(event, d) {{
                d.fx = event.x;
                d.fy = event.y;
            }}

            function dragended(event, d) {{
                if (!event.active) simulation.alphaTarget(0);
                d.fx = null;
                d.fy = null;
            }}

            // Control functions
            function zoomIn() {{
                svg.transition().call(zoom.scaleBy, 1.3);
            }}

            function zoomOut() {{
                svg.transition().call(zoom.scaleBy, 0.7);
            }}

            function resetView() {{
                svg.transition().call(zoom.transform, d3.zoomIdentity);
            }}

            function exportSVG() {{
                const svgElement = document.querySelector('svg');
                const serializer = new XMLSerializer();
                let source = serializer.serializeToString(svgElement);
                source = '<?xml version="1.0" encoding="UTF-8"?>' + source;

                const blob = new Blob([source], {{type: 'image/svg+xml'}});
                const url = URL.createObjectURL(blob);
                const link = document.createElement('a');
                link.href = url;
                link.download = 'knowledge_graph.svg';
                link.click();
                URL.revokeObjectURL(url);
            }}

            // Update stats
            function updateStats() {{
                const stats = `Nodes: ${{graphData.nodes.length}}<br>
                              Edges: ${{graphData.links.length}}`;
                document.getElementById('stats').innerHTML = stats;
            }}
            updateStats();
        </script>
    </body>
    </html>
    """

    return html_content

In [2]:
# Create a sample graph
G = nx.Graph()
G.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D'), ('D', 'E')])

# Define node sizes (you might calculate this based on centrality or other metrics)
node_sizes = {'A': 10, 'B': 8, 'C': 6, 'D': 12, 'E': 4}

# Generate the HTML
html_content = create_d3_html(G, node_sizes)

# Display in Colab
display(HTML(html_content))